In [ ]:
import re
import pandas as pd
import numpy as np
import difflib

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import pearsonr, spearmanr

import evaluate

############################################
# CONFIG
############################################

# uh-mazing translations are now sourced directly from the curated annotator
# pipeline (data/prolific_responses_filtered.csv) rather than the deleted
# data/uh-mazing.csv. We pivot the long-format CSV into the same wide layout
# the rest of this notebook expects (one column per LANG).
FILTERED_PATH = "../../data/prolific_responses_filtered.csv"
GOLD_PATH     = "../../data/uh-mazing-gold-irr.csv"

ID_COL = "ID"

LANGS = ["ZH", "ES", "HI", "FR", "DE", "IT", "CS", "AR"]

# POLICY:
# uh-mazing → prefer *_disfluent, else LANG
UHM_PRIORITY_SUFFIXES = ["_disfluent", ""]

OUT_PER_ITEM = "uhmazing_vs_gold_per_item.csv"
OUT_SUMMARY  = "uhmazing_vs_gold_summary.csv"

############################################
# HELPERS
############################################

def teleki_diff(a: str, b: str) -> float:
    """Teleki et al. string disagreement."""
    return difflib.SequenceMatcher(None, a, b).ratio()

def pick_uhm_col(cols, lang):
    """Prefer *_disfluent over bare LANG."""
    for suf in UHM_PRIORITY_SUFFIXES:
        name = f"{lang}{suf}"
        if name in cols:
            return name
    return None

def pick_gold_col(cols, lang):
    """Gold file should ONLY have bare LANG."""
    return lang if lang in cols else None

def safe_corr(x, y, method="pearson"):
    if len(x) < 3:
        return np.nan
    try:
        if method == "pearson":
            return pearsonr(x, y)[0]
        return spearmanr(x, y)[0]
    except Exception:
        return np.nan

def load_uhm_wide(filtered_csv_path: str) -> pd.DataFrame:
    """Pivot prolific_responses_filtered.csv into the wide LANG_disfluent layout
    that this notebook (and the old uh-mazing.csv) used.

    Each utterance has exactly one annotator's translation per language, so
    the pivot is unambiguous. Output columns: ID + {LANG}_disfluent for each
    language present.
    """
    pf = pd.read_csv(filtered_csv_path)
    pf = pf.copy()
    # Extract utt_id from question text like "[sw02005_A_111] Translation 14"
    pf["ID"] = pf["question"].astype(str).str.extract(r"\[(sw\d+_[AB]_\d+)\]")
    pf = pf.dropna(subset=["ID"]).copy()
    # Normalize sw02005 → sw2005 to match the Switchboard-style IDs used
    # in data/uh-mazing-gold-irr.csv.
    pf["ID"] = pf["ID"].str.replace(r"^sw0+(\d+)_", r"sw\1_", regex=True)
    # UID is "<LANG>_T<n>"; we only want the LANG part
    pf["LANG"] = pf["UID"].astype(str).str.split("_", n=1).str[0]
    wide = pf.pivot_table(
        index="ID", columns="LANG", values="answer", aggfunc="first"
    ).reset_index()
    wide.columns = [c if c == "ID" else f"{c}_disfluent" for c in wide.columns]
    return wide

############################################
# LOAD + JOIN
############################################

uhm = load_uhm_wide(FILTERED_PATH)
gold = pd.read_csv(GOLD_PATH)

if ID_COL not in uhm.columns or ID_COL not in gold.columns:
    raise ValueError("Both DataFrames must contain an ID column")

merged = uhm.merge(gold, on=ID_COL, how="inner", suffixes=("_uhm", "_gold"))

############################################
# MODELS
############################################

embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
)

############################################
# METRIC COMPUTATION
############################################

rows = []

for lang in LANGS:
    uhm_col  = pick_uhm_col(merged.columns, lang)
    gold_col = pick_gold_col(merged.columns, lang)

    if uhm_col is None or gold_col is None:
        continue

    sub = merged[[ID_COL, uhm_col, gold_col]].copy()

    # Strict filtering (Series-safe, no index bugs)
    sub = sub[
        sub[uhm_col].notna()
        & sub[gold_col].notna()
        & sub[uhm_col].astype(str).str.strip().ne("").replace("_","")
        & sub[gold_col].astype(str).str.strip().ne("").replace("_","")
    ].copy()

    if sub.empty:
        continue

    a_texts = sub[uhm_col].astype(str).tolist()   # uh-mazing
    b_texts = sub[gold_col].astype(str).tolist()  # gold

    # Teleki IRR
    teleki = np.array([
        teleki_diff(a, b) for a, b in zip(a_texts, b_texts)
    ])

    # Embedding IRR
    a_emb = embedder.encode(a_texts, normalize_embeddings=True)
    b_emb = embedder.encode(b_texts, normalize_embeddings=True)
    emb_irr = cosine_similarity(a_emb, b_emb).diagonal()

    for rid, t, e in zip(
        sub[ID_COL], teleki, emb_irr
    ):
        rows.append({
            "ID": rid,
            "language": lang,
            "uhm_col": uhm_col,
            "gold_col": gold_col,
            "teleki_irr": float(t),
            "embedding_irr": float(e), 
        })

############################################
# OUTPUT
############################################

results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError("No rows produced — check ID overlap and language columns")

summary = (
    results
    .groupby("language")
    .apply(lambda g: pd.Series({
        "n": len(g),
        "teleki_irr_mean": g["teleki_irr"].mean(),
        "embedding_irr_mean": g["embedding_irr"].mean(),
        "teleki_vs_embedding_pearson":
            safe_corr(g["teleki_irr"], g["embedding_irr"], "pearson"),
        "teleki_vs_embedding_spearman":
            safe_corr(g["teleki_irr"], g["embedding_irr"], "spearman"),
    }))
    .reset_index()
)

results.to_csv(OUT_PER_ITEM, index=False)
summary.to_csv(OUT_SUMMARY, index=False)

print("Wrote:", OUT_PER_ITEM)
print("Wrote:", OUT_SUMMARY)
print("\n=== SUMMARY ===")
display(summary.round(2))
display(summary.describe().round(2))
